# 09 — Sekans Özellik Ablasyon Çalışması

**Amaç**: Notebook 07'nin aynı FE pipeline'ı (in-silico olmadan) üzerinde,
sekans kökenli özellik gruplarını tek tek çıkararak F1 / AUC değişimini ölçmek.

**Ablasyon senaryoları** (her biri tam eğitim: 80/20 hold-out + 3-fold CV grid search):
1. Baseline — tüm özellikler
2. Tüm sekans çıkarıldı (OHE + k-mer)
3. Tüm OHE çıkarıldı
4. Tüm k-mer çıkarıldı
5. OHE → ref_base çıkarıldı (4 col)
6. OHE → alt_base çıkarıldı (4 col)
7. OHE → ref_amino çıkarıldı (20 col)
8. OHE → alt_amino çıkarıldı (20 col)
9. k-mer → DNA_11mer_Ref çıkarıldı (16 col)
10. k-mer → DNA_11mer_Alt çıkarıldı (16 col)
11. k-mer → Prot_11mer_Ref çıkarıldı (~430 col)
12. k-mer → Prot_11mer_Alt çıkarıldı (~434 col)

**Model**: LightGBM (12-combo grid search, en hızlı + en iyi F1)
**Panel**: General + Hereditary_Cancer

In [1]:
# Cell 1: Imports & Config
import sys, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, DATA_PATH, PANELS_SINGLE, RESULTS_NO_INSIL_DIR, REPORTS_DIR
from src.features import prepare_data_v3_no_insil
from src.models import grid_search_lightgbm
from src.metrics import compute_all_metrics

ABLATION_DIR = os.path.join(PROJECT_ROOT, 'results', 'ablation_sequence')
os.makedirs(ABLATION_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f'Proje koku: {PROJECT_ROOT}')
print(f'Ablasyon sonuc dizini: {ABLATION_DIR}')

Proje koku: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model
Ablasyon sonuc dizini: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\results\ablation_sequence


In [2]:
# Cell 2: Veri Yükleme & Feature Engineering
df_raw = pd.read_csv(DATA_PATH)
print(f'Ham veri: {df_raw.shape}')

# Target derivation (clinvar__sig mapping)
sig = df_raw['clinvar__sig'].str.lower().str.strip()
target_map = {
    'benign': 0, 'likely benign': 0,
    'pathogenic': 1, 'likely pathogenic': 1,
}
df_raw['target'] = sig.map(target_map)
df_raw = df_raw.dropna(subset=['target'])
df_raw['target'] = df_raw['target'].astype(int)
print(f'Target dagilimi: {df_raw["target"].value_counts().to_dict()}')

panel_series = df_raw['Panel'].copy()

# FE (in-silico olmadan) — NB07 ile aynı
df_v3 = prepare_data_v3_no_insil(df_raw)
df_v3['Panel'] = panel_series.values

print(f'\nFE sonrasi toplam boyut: {df_v3.shape}')
print(f'Panel dagilimi:\n{df_v3["Panel"].value_counts()}')

Ham veri: (4287, 119)
Target dagilimi: {1: 2951, 0: 1336}
  71 in-silico sutunu droplandi
  OHE (ayri alfabe): 48 feature
  K-mer DNA_11mer_Ref: 16 2-mer feature
  K-mer DNA_11mer_Alt: 16 2-mer feature
  K-mer Prot_11mer_Ref: 430 2-mer feature
  K-mer Prot_11mer_Alt: 434 2-mer feature
  24 gereksiz sutun kaldirildi
  r>0.99 kopya filtreleme: 14 sutun dusuruldu
[V3-NoInSil] Veri boyutu: 4287 x 954

FE sonrasi toplam boyut: (4287, 954)
Panel dagilimi:
Panel
General              3156
Hereditary_Cancer     715
PAH                   324
CFTR                   92
Name: count, dtype: int64


In [3]:
# Cell 3: Sekans Özellik Gruplarını Tanımla

all_cols = df_v3.columns.tolist()

# OHE grupları (base__ref_base__{A/C/G/T}, base__alt_base__{A/C/G/T},
#               ref_amino__{A..Y}, alt_amino__{A..Y})
ohe_ref_base  = [c for c in all_cols if c.startswith('base__ref_base__')]
ohe_alt_base  = [c for c in all_cols if c.startswith('base__alt_base__')]
ohe_ref_amino = [c for c in all_cols if c.startswith('ref_amino__')]
ohe_alt_amino = [c for c in all_cols if c.startswith('alt_amino__')]
all_ohe = ohe_ref_base + ohe_alt_base + ohe_ref_amino + ohe_alt_amino

# k-mer grupları (DNA_11mer_Ref__2mer_*, DNA_11mer_Alt__2mer_*,
#                 Prot_11mer_Ref__2mer_*, Prot_11mer_Alt__2mer_*)
kmer_dna_ref  = [c for c in all_cols if c.startswith('DNA_11mer_Ref__2mer_')]
kmer_dna_alt  = [c for c in all_cols if c.startswith('DNA_11mer_Alt__2mer_')]
kmer_prot_ref = [c for c in all_cols if c.startswith('Prot_11mer_Ref__2mer_')]
kmer_prot_alt = [c for c in all_cols if c.startswith('Prot_11mer_Alt__2mer_')]
all_kmer = kmer_dna_ref + kmer_dna_alt + kmer_prot_ref + kmer_prot_alt

all_seq = all_ohe + all_kmer

print('=== SEKANS ÖZELLİK GRUPLARI ===')
print(f'OHE - ref_base   : {len(ohe_ref_base):4d} sutun')
print(f'OHE - alt_base   : {len(ohe_alt_base):4d} sutun')
print(f'OHE - ref_amino  : {len(ohe_ref_amino):4d} sutun')
print(f'OHE - alt_amino  : {len(ohe_alt_amino):4d} sutun')
print(f'  Toplam OHE     : {len(all_ohe):4d} sutun')
print()
print(f'k-mer - DNA_Ref  : {len(kmer_dna_ref):4d} sutun')
print(f'k-mer - DNA_Alt  : {len(kmer_dna_alt):4d} sutun')
print(f'k-mer - Prot_Ref : {len(kmer_prot_ref):4d} sutun')
print(f'k-mer - Prot_Alt : {len(kmer_prot_alt):4d} sutun')
print(f'  Toplam k-mer   : {len(all_kmer):4d} sutun')
print()
print(f'Toplam sekans    : {len(all_seq):4d} sutun')
print(f'Toplam feature   : {df_v3.shape[1] - 2:4d} sutun (target+Panel haric)')

# Panel belirleme (NB07 ile aynı mantık)
TRAIN_PANELS = []
TEST_ONLY_PANELS = []
for p in PANELS_SINGLE:
    p_df = df_v3[df_v3['Panel'] == p]
    vc = p_df['target'].value_counts()
    if vc.min() >= 5:
        TRAIN_PANELS.append(p)
    else:
        TEST_ONLY_PANELS.append(p)

print(f'\nEgitim panelleri: {TRAIN_PANELS}')

# Ablasyon senaryoları: (isim, kaldırılacak sütunlar listesi)
ABLATION_SCENARIOS = [
    ('Baseline (tum ozellikler)',    []),
    ('Tum sekans kaldirildi',        all_seq),
    ('Tum OHE kaldirildi',           all_ohe),
    ('Tum k-mer kaldirildi',         all_kmer),
    ('OHE-ref_base kaldirildi',      ohe_ref_base),
    ('OHE-alt_base kaldirildi',      ohe_alt_base),
    ('OHE-ref_amino kaldirildi',     ohe_ref_amino),
    ('OHE-alt_amino kaldirildi',     ohe_alt_amino),
    ('kmer-DNA_Ref kaldirildi',      kmer_dna_ref),
    ('kmer-DNA_Alt kaldirildi',      kmer_dna_alt),
    ('kmer-Prot_Ref kaldirildi',     kmer_prot_ref),
    ('kmer-Prot_Alt kaldirildi',     kmer_prot_alt),
]

print(f'\nToplam {len(ABLATION_SCENARIOS)} ablasyon senaryosu')

=== SEKANS ÖZELLİK GRUPLARI ===
OHE - ref_base   :    4 sutun
OHE - alt_base   :    4 sutun
OHE - ref_amino  :   20 sutun
OHE - alt_amino  :   20 sutun
  Toplam OHE     :   48 sutun

k-mer - DNA_Ref  :   16 sutun
k-mer - DNA_Alt  :   16 sutun
k-mer - Prot_Ref :  429 sutun
k-mer - Prot_Alt :  422 sutun
  Toplam k-mer   :  883 sutun

Toplam sekans    :  931 sutun
Toplam feature   :  952 sutun (target+Panel haric)

Egitim panelleri: ['General', 'Hereditary_Cancer']

Toplam 12 ablasyon senaryosu


In [4]:
# Cell 4: Ablasyon Döngüsü (LightGBM, her panel × her senaryo)

ablation_results = []

for panel in TRAIN_PANELS:
    print(f"\n{'='*70}")
    print(f"PANEL: {panel}")
    print(f"{'='*70}")

    df_panel_full = df_v3[df_v3['Panel'] == panel].drop(columns=['Panel'])
    X_all = df_panel_full.drop(columns=['target'])
    y = df_panel_full['target']

    # NB07 ile birebir aynı split (SEED=42)
    X_cv, X_holdout, y_cv, y_holdout = train_test_split(
        X_all, y, test_size=0.20, random_state=SEED, stratify=y
    )
    print(f'  CV: {len(y_cv)} | Hold-out: {len(y_holdout)}')

    for scenario_name, drop_cols in ABLATION_SCENARIOS:
        # Bu senaryoda kaldırılacak sütunlar (mevcut olanları filtrele)
        actual_drop = [c for c in drop_cols if c in X_all.columns]
        n_dropped = len(actual_drop)
        n_remaining = X_all.shape[1] - n_dropped

        X_cv_s = X_cv.drop(columns=actual_drop)
        X_ho_s = X_holdout.drop(columns=actual_drop)

        t0 = time.time()
        try:
            model, best_combo, best_thr, y_prob_ho = grid_search_lightgbm(
                X_cv_s, y_cv, X_ho_s, y_holdout
            )
            y_pred_ho = (y_prob_ho >= best_thr).astype(int)
            metrics = compute_all_metrics(y_holdout, y_pred_ho, y_prob_ho)
            elapsed = time.time() - t0

            row = {
                'panel': panel,
                'senaryo': scenario_name,
                'n_kaldirildi': n_dropped,
                'n_kalan': n_remaining,
                'f1': metrics['f1'],
                'auc_roc': metrics['auc_roc'],
                'auc_pr': metrics['auc_pr'],
                'precision': metrics['precision'],
                'recall': metrics['recall'],
                'mcc': metrics['mcc'],
                'elapsed_sec': round(elapsed, 1),
                'best_params': str(best_combo),
            }
        except Exception as e:
            row = {
                'panel': panel, 'senaryo': scenario_name,
                'n_kaldirildi': n_dropped, 'n_kalan': n_remaining,
                'f1': np.nan, 'auc_roc': np.nan, 'auc_pr': np.nan,
                'precision': np.nan, 'recall': np.nan, 'mcc': np.nan,
                'elapsed_sec': round(time.time() - t0, 1),
                'best_params': f'HATA: {e}',
            }
            print(f"    HATA [{scenario_name}]: {e}")

        ablation_results.append(row)
        print(f"  [{panel}] {scenario_name:<35s} "
              f"F1={row['f1']:.4f}  AUC={row['auc_roc']:.4f}  "
              f"kalan={n_remaining}  ({row['elapsed_sec']:.0f}s)")

# CSV kaydet
abl_df = pd.DataFrame(ablation_results)
abl_df.to_csv(os.path.join(ABLATION_DIR, 'ablation_results.csv'), index=False)
print(f'\nSonuclar kaydedildi: {ABLATION_DIR}/ablation_results.csv')


PANEL: General
  CV: 2524 | Hold-out: 632
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 100, 'num_leaves': 127, 'learning_rate': 0.1} -> CV F1=0.9505
  [General] Baseline (tum ozellikler)           F1=0.9465  AUC=0.9674  kalan=952  (26s)
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 100, 'num_leaves': 127, 'learning_rate': 0.05} -> CV F1=0.9461
  [General] Tum sekans kaldirildi               F1=0.9409  AUC=0.9563  kalan=21  (14s)
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 200, 'num_leaves': 127, 'learning_rate': 0.05} -> CV F1=0.9520
  [General] Tum OHE kaldirildi                  F1=0.9468  AUC=0.9677  kalan=904  (22s)
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators': 100, 'num_leaves': 63, 'learning_rate': 0.05} -> CV F1=0.9488
  [General] Tum k-mer kaldirildi                F1=0.9412  AUC=0.9618  kalan=69  (14s)
  LightGBM Grid Search: 12 kombinasyon
  En iyi combo: {'n_estimators

In [5]:
# Cell 5: Sonuç Tablosu — Baseline'a göre delta F1
from IPython.display import display

abl_df = pd.read_csv(os.path.join(ABLATION_DIR, 'ablation_results.csv'))

print('=== ABLASYON SONUÇLARI — TAM TABLO ===')

for panel in TRAIN_PANELS:
    panel_abl = abl_df[abl_df['panel'] == panel].copy()

    # Baseline F1 değerini al
    baseline_row = panel_abl[panel_abl['senaryo'] == 'Baseline (tum ozellikler)']
    if len(baseline_row) == 0:
        continue
    baseline_f1   = baseline_row['f1'].values[0]
    baseline_auc  = baseline_row['auc_roc'].values[0]

    panel_abl['delta_f1']  = panel_abl['f1']      - baseline_f1
    panel_abl['delta_auc'] = panel_abl['auc_roc'] - baseline_auc

    # Renk: pozitif delta (iyileşme, mavi), negatif delta (bozulma, kırmızı)
    print(f'\n--- {panel} (Baseline F1={baseline_f1:.4f}, AUC={baseline_auc:.4f}) ---')
    disp_cols = ['senaryo', 'n_kaldirildi', 'n_kalan', 'f1', 'auc_roc', 'precision', 'recall', 'delta_f1', 'delta_auc']
    display(panel_abl[disp_cols].round(4).reset_index(drop=True))

=== ABLASYON SONUÇLARI — TAM TABLO ===

--- General (Baseline F1=0.9465, AUC=0.9674) ---


,senaryo,n_kaldirildi,n_kalan,f1,auc_roc,precision,recall,delta_f1,delta_auc
0,Baseline (tum ozellikler),0,952,0.9465,0.9674,0.9219,0.9725,0.0000,0.0000
1,Tum sekans kaldirildi,931,21,0.9409,0.9563,0.9174,0.9657,-0.0056,-0.0111
2,Tum OHE kaldirildi,48,904,0.9468,0.9677,0.9183,0.9771,0.0002,0.0003
3,Tum k-mer kaldirildi,883,69,0.9412,0.9618,0.9138,0.9703,-0.0054,-0.0056
4,OHE-ref_base kaldirildi,4,948,0.9446,0.9681,0.9330,0.9565,-0.0019,0.0007
5,OHE-alt_base kaldirildi,4,948,0.9475,0.9704,0.9258,0.9703,0.0009,0.0029
6,OHE-ref_amino kaldirildi,20,932,0.9473,0.9681,0.9295,0.9657,0.0007,0.0007
7,OHE-alt_amino kaldirildi,20,932,0.9451,0.9705,0.9254,0.9657,-0.0014,0.0031
8,kmer-DNA_Ref kaldirildi,16,936,0.9455,0.9708,0.9199,0.9725,-0.0011,0.0034
9,kmer-DNA_Alt kaldirildi,16,936,0.9465,0.9683,0.9219,0.9725,0.0000,0.0009



--- Hereditary_Cancer (Baseline F1=0.9362, AUC=0.9810) ---


,senaryo,n_kaldirildi,n_kalan,f1,auc_roc,precision,recall,delta_f1,delta_auc
0,Baseline (tum ozellikler),0,952,0.9362,0.9810,0.9429,0.9296,0.0000,0.0000
1,Tum sekans kaldirildi,931,21,0.9504,0.9830,0.9571,0.9437,0.0142,0.0020
2,Tum OHE kaldirildi,48,904,0.9353,0.9814,0.9559,0.9155,-0.0009,0.0004
3,Tum k-mer kaldirildi,883,69,0.9429,0.9785,0.9565,0.9296,0.0067,-0.0025
4,OHE-ref_base kaldirildi,4,948,0.9429,0.9824,0.9565,0.9296,0.0067,0.0014
5,OHE-alt_base kaldirildi,4,948,0.9315,0.9816,0.9067,0.9577,-0.0047,0.0006
6,OHE-ref_amino kaldirildi,20,932,0.9429,0.9830,0.9565,0.9296,0.0067,0.0020
7,OHE-alt_amino kaldirildi,20,932,0.9353,0.9816,0.9559,0.9155,-0.0009,0.0006
8,kmer-DNA_Ref kaldirildi,16,936,0.9286,0.9797,0.9420,0.9155,-0.0076,-0.0014
9,kmer-DNA_Alt kaldirildi,16,936,0.9504,0.9832,0.9571,0.9437,0.0142,0.0022


In [6]:
# Cell 6: Görselleştirme
fig_paths = []

# --- 1. Delta F1 Bar Chart (her panel için) ---
for panel in TRAIN_PANELS:
    panel_abl = abl_df[abl_df['panel'] == panel].copy()
    baseline_f1 = panel_abl[panel_abl['senaryo'] == 'Baseline (tum ozellikler)']['f1'].values[0]
    panel_abl['delta_f1'] = panel_abl['f1'] - baseline_f1

    # Baseline satırını ayır, sıralamayı düzenle
    non_baseline = panel_abl[panel_abl['senaryo'] != 'Baseline (tum ozellikler)'].sort_values('delta_f1')

    colors = ['#F44336' if d < 0 else '#4CAF50' for d in non_baseline['delta_f1']]

    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(non_baseline['senaryo'], non_baseline['delta_f1'], color=colors)
    ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
    ax.set_title(f'{panel} — Sekans Grubu Çıkarma Etkisi (ΔF1 vs Baseline={baseline_f1:.4f})', fontsize=13)
    ax.set_xlabel('ΔF1 (pozitif = iyileşme, negatif = bozulma)')
    ax.set_ylabel('Ablasyon Senaryosu')
    plt.tight_layout()
    p = os.path.join(ABLATION_DIR, f'{panel}_delta_f1.png')
    fig.savefig(p, dpi=150)
    fig_paths.append(p)
    plt.show()

# --- 2. Tüm paneller: F1 karşılaştırma heatmap ---
pivot_f1 = abl_df.pivot(index='senaryo', columns='panel', values='f1')

# Satır sıralama: baseline üste, geri kalanlar delta_f1'e göre
if TRAIN_PANELS:
    ref_panel = TRAIN_PANELS[0]
    baseline_f1_ref = pivot_f1.loc['Baseline (tum ozellikler)', ref_panel]
    pivot_f1['_delta'] = pivot_f1[ref_panel] - baseline_f1_ref
    pivot_f1 = pivot_f1.sort_values('_delta').drop(columns=['_delta'])
    # Baseline'ı en alta taşı
    bl_row = pivot_f1.loc[pivot_f1.index == 'Baseline (tum ozellikler)']
    rest = pivot_f1.loc[pivot_f1.index != 'Baseline (tum ozellikler)']
    pivot_f1 = pd.concat([bl_row, rest])

fig, ax = plt.subplots(figsize=(8, len(pivot_f1) * 0.65 + 2))
sns.heatmap(pivot_f1.astype(float), annot=True, fmt='.4f', cmap='RdYlGn',
            vmin=pivot_f1.min().min() - 0.01, vmax=pivot_f1.max().max() + 0.01,
            ax=ax, linewidths=0.5)
ax.set_title('Ablasyon F1 Heatmap (Panel × Senaryo)', fontsize=13)
ax.set_xlabel('Panel')
ax.set_ylabel('Ablasyon Senaryosu')
plt.tight_layout()
p = os.path.join(ABLATION_DIR, 'ablation_f1_heatmap.png')
fig.savefig(p, dpi=150)
fig_paths.append(p)
plt.show()

# --- 3. Özellik sayısı vs F1 scatter (hangi grup ne kadar özellik katkısı) ---
fig, axes = plt.subplots(1, len(TRAIN_PANELS), figsize=(7 * len(TRAIN_PANELS), 6))
if len(TRAIN_PANELS) == 1:
    axes = [axes]
for ax, panel in zip(axes, TRAIN_PANELS):
    panel_abl = abl_df[abl_df['panel'] == panel].copy()
    baseline_f1 = panel_abl[panel_abl['senaryo'] == 'Baseline (tum ozellikler)']['f1'].values[0]
    panel_abl['delta_f1'] = panel_abl['f1'] - baseline_f1

    non_bl = panel_abl[panel_abl['senaryo'] != 'Baseline (tum ozellikler)']
    sc = ax.scatter(non_bl['n_kaldirildi'], non_bl['delta_f1'],
                    s=80, c=non_bl['delta_f1'], cmap='RdYlGn', zorder=3)
    for _, row in non_bl.iterrows():
        label = row['senaryo'].replace(' kaldirildi', '').replace('Tum ', 'Tüm ')
        ax.annotate(label, (row['n_kaldirildi'], row['delta_f1']),
                    textcoords='offset points', xytext=(5, 2), fontsize=7.5)
    ax.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax.set_title(f'{panel} — Kaldırılan Özellik Sayısı vs ΔF1', fontsize=11)
    ax.set_xlabel('Kaldırılan sütun sayısı')
    ax.set_ylabel('ΔF1')
    plt.colorbar(sc, ax=ax, label='ΔF1')
plt.tight_layout()
p = os.path.join(ABLATION_DIR, 'feature_count_vs_delta_f1.png')
fig.savefig(p, dpi=150)
fig_paths.append(p)
plt.show()

print(f'\nToplam {len(fig_paths)} grafik kaydedildi.')


Toplam 4 grafik kaydedildi.


In [7]:
# Cell 7: Özet Tablo — Baskı için temiz format
print('=== SEKANS ABLASYON ÖZET TABLOSU ===')
print('(Her senaryoda kaldırılan sutun sayısı ve delta F1)')
print()

for panel in TRAIN_PANELS:
    panel_abl = abl_df[abl_df['panel'] == panel].copy()
    bl_f1  = panel_abl[panel_abl['senaryo'] == 'Baseline (tum ozellikler)']['f1'].values[0]
    bl_auc = panel_abl[panel_abl['senaryo'] == 'Baseline (tum ozellikler)']['auc_roc'].values[0]
    panel_abl['delta_f1']  = panel_abl['f1']      - bl_f1
    panel_abl['delta_auc'] = panel_abl['auc_roc'] - bl_auc

    print(f'Panel: {panel}  |  Baseline F1={bl_f1:.4f}, AUC={bl_auc:.4f}')
    print(f'{"Senaryo":<40} {"Kaldırılan":>10} {"F1":>7} {"ΔF1":>7} {"AUC":>7} {"ΔAUC":>7}')
    print('-' * 80)
    for _, row in panel_abl.iterrows():
        marker = ' <-- BASELINE' if 'Baseline' in row['senaryo'] else ''
        print(f"{row['senaryo']:<40} {int(row['n_kaldirildi']):>10} "
              f"{row['f1']:>7.4f} {row['delta_f1']:>+7.4f} "
              f"{row['auc_roc']:>7.4f} {row['delta_auc']:>+7.4f}{marker}")
    print()

# Sonucu CSV olarak da yaz (delta sütunlarıyla)
rows = []
for panel in TRAIN_PANELS:
    panel_abl = abl_df[abl_df['panel'] == panel].copy()
    bl_f1  = panel_abl[panel_abl['senaryo'] == 'Baseline (tum ozellikler)']['f1'].values[0]
    bl_auc = panel_abl[panel_abl['senaryo'] == 'Baseline (tum ozellikler)']['auc_roc'].values[0]
    panel_abl['delta_f1']  = panel_abl['f1']      - bl_f1
    panel_abl['delta_auc'] = panel_abl['auc_roc'] - bl_auc
    rows.append(panel_abl)

summary_df = pd.concat(rows, ignore_index=True)
summary_df.to_csv(os.path.join(ABLATION_DIR, 'ablation_summary_with_delta.csv'), index=False)
print(f'Delta tablosu kaydedildi: {ABLATION_DIR}/ablation_summary_with_delta.csv')

=== SEKANS ABLASYON ÖZET TABLOSU ===
(Her senaryoda kaldırılan sutun sayısı ve delta F1)

Panel: General  |  Baseline F1=0.9465, AUC=0.9674
Senaryo                                  Kaldırılan      F1     ΔF1     AUC    ΔAUC
--------------------------------------------------------------------------------
Baseline (tum ozellikler)                         0  0.9465 +0.0000  0.9674 +0.0000 <-- BASELINE
Tum sekans kaldirildi                           931  0.9409 -0.0056  0.9563 -0.0111
Tum OHE kaldirildi                               48  0.9468 +0.0002  0.9677 +0.0003
Tum k-mer kaldirildi                            883  0.9412 -0.0054  0.9618 -0.0056
OHE-ref_base kaldirildi                           4  0.9446 -0.0019  0.9681 +0.0007
OHE-alt_base kaldirildi                           4  0.9475 +0.0009  0.9704 +0.0029
OHE-ref_amino kaldirildi                         20  0.9473 +0.0007  0.9681 +0.0007
OHE-alt_amino kaldirildi                         20  0.9451 -0.0014  0.9705 +0.0031
kmer-DNA_R

In [10]:
# Cell 8: PDF Rapor
from fpdf import FPDF
from datetime import datetime


class AblationReport(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 10)
        self.cell(0, 8, 'Teknofest - Sekans Ozellik Ablasyon Raporu', align='C',
                  new_x='LMARGIN', new_y='NEXT')
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.cell(0, 10, f'Sayfa {self.page_no()}/{{nb}}', align='C')


pdf = AblationReport()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=20)

# Baslik
pdf.add_page()
pdf.set_font('Helvetica', 'B', 18)
pdf.ln(30)
pdf.cell(0, 14, 'Sekans Ozellik Ablasyon Raporu', align='C', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 12)
pdf.cell(0, 9, 'Model: LightGBM  |  FE: v3_no_insil  |  Split: 80/20', align='C',
          new_x='LMARGIN', new_y='NEXT')
pdf.cell(0, 9, f'Tarih: {datetime.now().strftime("%Y-%m-%d %H:%M")}', align='C',
          new_x='LMARGIN', new_y='NEXT')
pdf.ln(8)
pdf.set_font('Helvetica', '', 10)
pdf.cell(0, 7, f'Sekans gruplari: OHE ({len(all_ohe)} col), k-mer ({len(all_kmer)} col), Toplam={len(all_seq)}',
          align='C', new_x='LMARGIN', new_y='NEXT')

# Ozet tablo sayfasi
for panel in TRAIN_PANELS:
    pdf.add_page()
    pdf.set_font('Helvetica', 'B', 13)
    panel_abl = summary_df[summary_df['panel'] == panel]
    bl_f1 = panel_abl[panel_abl['senaryo'] == 'Baseline (tum ozellikler)']['f1'].values[0]
    pdf.cell(0, 9, f'{panel} Ablasyon Tablosu  (Baseline F1={bl_f1:.4f})', new_x='LMARGIN', new_y='NEXT')

    col_w = [62, 22, 20, 20, 20, 20, 20]
    hdrs  = ['Senaryo', 'Kaldirilan', 'F1', 'dF1', 'AUC', 'dAUC', 'Sure(s)']
    pdf.set_font('Helvetica', 'B', 8)
    for w, h in zip(col_w, hdrs):
        pdf.cell(w, 7, h, border=1, align='C')
    pdf.ln()
    pdf.set_font('Helvetica', '', 7)
    for _, row in panel_abl.iterrows():
        vals = [
            row['senaryo'],
            str(int(row['n_kaldirildi'])),
            f"{row['f1']:.4f}",
            f"{row['delta_f1']:+.4f}",
            f"{row['auc_roc']:.4f}",
            f"{row['delta_auc']:+.4f}",
            f"{row['elapsed_sec']:.0f}",
        ]
        for w, v in zip(col_w, vals):
            pdf.cell(w, 6, v, border=1, align='C')
        pdf.ln()

# Grafikler
for fp in fig_paths:
    if os.path.exists(fp):
        pdf.add_page()
        fname = os.path.basename(fp).replace('.png', '').replace('_', ' ').title()
        pdf.set_font('Helvetica', 'B', 11)
        pdf.cell(0, 9, fname, new_x='LMARGIN', new_y='NEXT')
        try:
            pdf.image(fp, x=10, w=190)
        except Exception as e:
            pdf.set_font('Helvetica', '', 9)
            pdf.cell(0, 7, f'Grafik yuklenemedi: {e}', new_x='LMARGIN', new_y='NEXT')

report_path = os.path.join(REPORTS_DIR, 'ablation_sequence_report.pdf')
pdf.output(report_path)
print(f'PDF rapor kaydedildi: {report_path}')


PDF rapor kaydedildi: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\reports\ablation_sequence_report.pdf
